# II. Genre Identification #

> **Objective**: This notebook main goal is to create a classification model that can predict accurately the musical genre of a song, based on its lyrics

## Grid Search Implementation ##

> In this step the model will be trained. To do so, different combinations of feature extraction and classification models were experimented. 


**Feature Extraction**: 
- Bag of Words
- TF-IDF
- Word2Vec
- Doc2Vec

**Classification Models**:
- Logistic Regression
- MultinomialNB
- Decision Trees
- Random Forest
- Neural Networks


**Step 1:** Importing the necessary libraries and the datasets.

In [19]:
# Import libraries
import pandas as pd
import numpy as np

import re
import matplotlib.pyplot as plt
import seaborn as sns

# models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score

# vectorizers

from sklearn import metrics
from sklearn.model_selection import StratifiedKFold
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.linear_model import LogisticRegression
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
# Progress Bar
from tqdm import tqdm
tqdm.pandas()

# Import preprocessing functions from utils.py
import utils

In [20]:
train_data = pd.read_csv('data/train_after_eda.csv')
train_data.head()

,id,title,tag,artist,views,lyrics,eda_lyrics
0,535805,Walk Away,rock,Tony Molina,699,When you said you loved me\nDid you mean it th...,said loved mean change mind one day pretend ca...
1,7519483,Gotta Make It Kid Naruto Rap,rap,Reece Lett,4,Kid Naruto Rap\n[Hook]\nEverybody wants you to...,kid naruto rap everybody wants hurt everybody ...
2,4892808,​this is what i asked for,pop,Elliot (DNK),389,[Verse 1]\nPeople tell me I've changed\nI find...,people tell changed find hard explain maybe fe...
3,1584150,Stealing Hearts,pop,Katie Armiger,126,You've been warned about me\nDon't try to get ...,warned try get close know want chain leave gho...
4,7639050,Get Ready,country,John Campbell Munro,2,[Verse 1]\nI can see the end is coming but I’v...,see end coming never liked odds need way even ...


**Step 2:** Grid Search

> In this step, the objective is to find the best combination of Feature Extractor and Classification model.

In [ ]:
# Train Test split
from sklearn.model_selection import train_test_split

import importlib
import utils
importlib.reload(utils)

X_train, X_test, y_train, y_test = train_test_split(train_data['eda_lyrics'], train_data['tag'], test_size=0.2, random_state=42)

models = ['LogisticRegression', 'MultinomialNB', 'DecisionTree', 'RandomForest', 'NeuralNetwork']

vectorizer_names = ['DictVectorizer', 'CountVectorizer', 'TfidfVectorizer', 'GloVe', 'Word2Vec', 'Doc2Vec']

stopword_options = ["none", "standard", "custom"]

combo, train_score, val_score = utils.grid_search(vectorizer_names, models, stopword_options, X_train, y_train)



Grid Search starting
90 combinations being tested
------------------------------------------------------------
Combination 0/90
Combination 1/90
Combination 2/90
Combination 3/90
Combination 4/90
Combination 5/90
Combination 6/90
Combination 7/90
Combination 8/90
Combination 9/90
Combination 10/90


In [ ]:


# Tagging documents
tagged_data = [TaggedDocument(words=doc.split(), tags=[str(i)]) for i, doc in enumerate(X_train)]

# Training Doc2Vec model
model = Doc2Vec(vector_size=300, window=5, min_count=1, workers=4, epochs=20)
model.build_vocab(tagged_data)
model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

# Generate vectors for training and testing sets
X_train_vectors = utils.infer_vectors(model, X_train)
X_test_vectors = utils.infer_vectors(model, X_test)


# Define the parameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # regularization strength
    'penalty': ['l1', 'l2'],  # penalty type
    'solver': ['liblinear', 'saga']  # solver algorithm
}

# Initialize logistic regression model
logreg_model = LogisticRegression(max_iter=1000)

# Initialize StratifiedKFold for cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize GridSearchCV with F1 score as the scoring metric
grid_search = GridSearchCV(logreg_model, param_grid, cv, scoring='f1_weighted')

# Fit the grid search to the data
grid_search.fit(X_train_vectors, y_train)

# Print the best hyperparameters found
print("Best hyperparameters:", grid_search.best_params_)

# Get the best model
best_model = grid_search.best_estimator_

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4140841165.py, line 38)